<a href="https://colab.research.google.com/github/Emmanuel-NS/FARUMASI-AI/blob/main/notebooks/train_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FARUMASI Pharmacist Assistant: full training and evaluation pipeline

Runtime → Change runtime type → **T4 GPU**. Run the cells in order (about 60–90 minutes in total).

Stages: data download → preparation → intent classifier (baseline + DistilBERT) → retrieval (BM25, MiniLM zero-shot, fine-tuned MiniLM, hybrid) → answer generation (extractive, Flan-T5 zero-shot, Flan-T5 + LoRA) → safety check → summary tables → (optional) deploy to Hugging Face Spaces.

In [17]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "https://github.com/Emmanuel-NS/FARUMASI-AI.git"
%cd /content
!rm -rf repo && git clone -q $REPO_URL repo
%cd /content/repo
!pip install -q -r requirements.txt
# Colab ships torchao 0.10, which recent peft rejects when creating LoRA layers; it is not used here.
!pip uninstall -y -q torchao
!python -c "import torch, transformers, peft, sentence_transformers as st; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| sentence-transformers', st.__version__)"

name, memory.total [MiB]
Tesla T4, 15360 MiB
/content/repo/repo


## 1. Data

In [18]:
!python -m pharmqa.data.download
!python -m pharmqa.data.prepare
!python -m pharmqa.data.intent_data

MedQuAD: 11274 XML documents -> /content/repo/repo/data/raw/medquad
Creating json from Arrow format: 100% 1/1 [00:00<00:00, 33.98ba/s]
MedicationQA: 690 rows
CLINC150: saved all splits
HealthCareMagic: saved 20k patient messages
openFDA: 100% 142/142 [01:28<00:00,  1.60it/s]
openFDA: kept labels for 140/142 medicines -> /content/repo/repo/data/raw/openfda_labels.jsonl
{
  "corpus_passages": 8840,
  "corpus_by_source": {
    "openFDA": 2049,
    "MedQuAD": 5546,
    "MedicationQA": 655,
    "RwandaFDA-PIL": 590
  },
  "qa_total": 9518,
  "qa_by_source": {
    "MedQuAD": 5580,
    "openFDA-templated": 2523,
    "RwandaFDA-PIL": 760,
    "MedicationQA": 655
  },
  "splits": {
    "train": {
      "n": 7200,
      "focus_entities": 1672,
      "by_source": {
        "MedQuAD": 4304,
        "openFDA-templated": 1893,
        "MedicationQA": 511,
        "RwandaFDA-PIL": 492
      }
    },
    "val": {
      "n": 1136,
      "focus_entities": 212,
      "by_source": {
        "MedQuAD": 663

## 2. Intent / scope classifier
Baseline TF-IDF + logistic regression (4 configs), then DistilBERT trained on the original data and on the data augmented with real patient questions.

In [19]:
!python -m pharmqa.intent.baseline
!python -m pharmqa.intent.transformer --train_file train --name distilbert_original
!python -m pharmqa.intent.transformer --train_file train_consumer --name distilbert

[tfidf_lr_word] test acc=0.9925 macroF1=0.9921 | challenge acc=0.3929 | by source={'CLINC150': 0.9883, 'MedQuAD/10_MPlus_ADAM_QA': 1.0, 'MedQuAD/11_MPlusDrugs_QA': 1.0, 'MedQuAD/12_MPlusHerbsSupplements_QA': 1.0, 'MedQuAD/1_CancerGov_QA': 1.0, 'MedQuAD/4_MPlus_Health_Topics_QA': 1.0, 'MedQuAD/5_NIDDK_QA': 1.0, 'MedQuAD/6_NINDS_QA': 1.0, 'MedQuAD/7_SeniorHealth_QA': 1.0, 'MedQuAD/8_NHLBI_QA_XML': 1.0, 'MedQuAD/9_CDC_QA': 1.0, 'MedicationQA': 0.5278}
[tfidf_lr_word_char] test acc=0.9947 macroF1=0.9943 | challenge acc=0.4643 | by source={'CLINC150': 0.9912, 'MedQuAD/10_MPlus_ADAM_QA': 1.0, 'MedQuAD/11_MPlusDrugs_QA': 1.0, 'MedQuAD/12_MPlusHerbsSupplements_QA': 1.0, 'MedQuAD/1_CancerGov_QA': 1.0, 'MedQuAD/4_MPlus_Health_Topics_QA': 1.0, 'MedQuAD/5_NIDDK_QA': 1.0, 'MedQuAD/6_NINDS_QA': 1.0, 'MedQuAD/7_SeniorHealth_QA': 1.0, 'MedQuAD/8_NHLBI_QA_XML': 1.0, 'MedQuAD/9_CDC_QA': 1.0, 'MedicationQA': 0.6944}
[tfidf_lr_word_consumer] test acc=0.9775 macroF1=0.9784 | challenge acc=0.7321 | by sourc

## 3. Retrieval

In [20]:
!python -m pharmqa.retrieval.bm25
!python -m pharmqa.retrieval.dense --mode all --epochs 2 --batch 64 --lr 2e-5

BM25: 100% 1182/1182 [00:10<00:00, 115.07it/s]
[bm25] recall@1=0.3223 recall@3=0.5728 recall@5=0.7056 recall@10=0.8316 mrr@10=0.478 topic@1=0.8519
/content/repo/repo/pharmqa/retrieval/dense.py:21: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss
/content/repo/repo/pharmqa/retrieval/dense.py:22: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers
Loading weights: 100% 103/103 [00:00<00:00, 4420.79it/s]
Batches: 100% 70/70 [00:22<00:00,  3.07it/s]
[minilm_zeroshot] recall@1=0.4357 recall@3=0.6988 recall@5=0.8037 recall@10=0.8976 mrr@10=0.5888 topic@1=0.8816
Loa

Rwanda FDA experiment (before): same retriever trained **without** the Rwanda leaflet questions. Compare the `RwandaFDA-PIL` rows in `results/retrieval_minilm_finetuned*.json`. Saved separately; the deployed retriever is untouched.

In [21]:
!python -m pharmqa.retrieval.dense --mode finetune --epochs 2 --batch 64 --lr 2e-5 --exclude_source RwandaFDA-PIL

/content/repo/repo/pharmqa/retrieval/dense.py:21: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss
/content/repo/repo/pharmqa/retrieval/dense.py:22: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers
Loading weights: 100% 103/103 [00:00<00:00, 5220.07it/s]
{'loss': '0.06646', 'grad_norm': '1.619', 'learning_rate': '1.695e-05', 'epoch': '0.4762'}
{'loss': '0.03277', 'grad_norm': '0.3291', 'learning_rate': '1.168e-05', 'epoch': '0.9524'}
 50% 105/210 [01:00<00:53,  1.95it/s]
  0% 0/17 [00:00<?, ?it/s]
 12% 2/17 [00:00<00:01,  9.86it/s]
 18% 3/17 [00:00<00:02, 

## 4. Answer generation (Flan-T5-base + LoRA)

LoRA training takes about 45-60 min on a T4. Short on time? Add `--max_train 4000 --epochs 2` (about 20 min).

**Fresh session, generation only:** run setup, section 1, then `!python -m pharmqa.retrieval.dense --mode finetune` (about 10 min, needed for the retrieved-context evaluation), then this section, 5 and 6.

In [26]:
!python -m pharmqa.generation.train_lora --epochs 3 --lr 3e-4 --rank 16 --batch 4 --grad_accum 4

hard negatives (train): 100% 7200/7200 [00:55<00:00, 129.50it/s]
Map: 100% 7200/7200 [00:10<00:00, 659.67 examples/s]
hard negatives (val): 100% 400/400 [00:03<00:00, 117.36it/s]
Map: 100% 400/400 [00:00<00:00, 681.19 examples/s]
Loading weights: 100% 282/282 [00:00<00:00, 4065.27it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/repo/repo/pharmqa/generation/train_lora.py", line 100, in <module>
    main()
    ~~~~^^
  File "/content/repo/repo/pharmqa/generation/train_lora.py", line 74, in main
    model = get_peft_model(model, LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM, r=args.ran

In [14]:
for system in ["extractive", "zeroshot", "lora"]:
    for context in ["gold", "retrieved"]:
        !python -m pharmqa.generation.evaluate --system {system} --context {context} --n 400

hard negatives: 100% 400/400 [00:04<00:00, 86.18it/s]
Loading weights: 100% 100/100 [00:00<00:00, 4554.03it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
{
  "system": "extractive",
  "context": "gold",
  "n": 400,
  "rougeL": 0.9329,
  "bertscore_f1": 0.9853,
  "avg_words": 60.5,
  "gold_in_context_rate": 1.0,
  "rougeL_when_gold_in_context": 0.9329,
  "by_source": {
    "MedQuAD": {
      "rougeL": 0.9572,
      "bertscore_f1": 0.9903
    },
    "MedicationQA": {
      "rougeL": 1.0,
      "bertscore_f1": 1.0
    },
    "RwandaFDA-PIL": {
 

Rwanda FDA experiment (before) for the reader: LoRA trained without Rwanda leaflet questions, evaluated on the same test questions.

In [22]:
!python -m pharmqa.generation.train_lora --epochs 3 --lr 3e-4 --rank 16 --batch 4 --grad_accum 4 --exclude_source RwandaFDA-PIL --out artifacts/generator_flant5_lora_no_rwandafda
!python -m pharmqa.generation.evaluate --system lora --context gold --n 400 --adapter artifacts/generator_flant5_lora_no_rwandafda --tag _no_rwandafda

hard negatives (train): 100% 6708/6708 [00:52<00:00, 127.08it/s]
Map: 100% 6708/6708 [00:10<00:00, 668.50 examples/s]
hard negatives (val): 100% 400/400 [00:03<00:00, 112.28it/s]
Map: 100% 400/400 [00:00<00:00, 652.18 examples/s]
Loading weights: 100% 282/282 [00:00<00:00, 3999.41it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/repo/repo/pharmqa/generation/train_lora.py", line 100, in <module>
    main()
    ~~~~^^
  File "/content/repo/repo/pharmqa/generation/train_lora.py", line 74, in main
    model = get_peft_model(model, LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM, r=args.ran

## 5. Safety check and summary tables

In [23]:
!python -m pharmqa.safety_eval
!python -m pharmqa.report_tables

{
  "accuracy": 1.0,
  "errors": []
}
# Experiment summary

## Dataset

- Corpus passages: 8840 ({'openFDA': 2049, 'MedQuAD': 5546, 'MedicationQA': 655, 'RwandaFDA-PIL': 590})
- QA pairs: 9518 ({'MedQuAD': 5580, 'openFDA-templated': 2523, 'RwandaFDA-PIL': 760, 'MedicationQA': 655})
- train: 7200 questions, 1672 distinct medicines/conditions
- val: 1136 questions, 212 distinct medicines/conditions
- test: 1182 questions, 233 distinct medicines/conditions

## Intent classification

| model                       |   test acc |   test macro-F1 |   MedicationQA acc |   challenge acc |   challenge macro-F1 |
|:----------------------------|-----------:|----------------:|-------------------:|----------------:|---------------------:|
| distilbert                  |     0.9941 |          0.9938 |             0.8889 |          0.8036 |               0.7822 |
| distilbert_original         |     0.998  |          0.9978 |             0.9167 |          0.5536 |               0.486  |
| tfidf_lr_word

## 6. Save results and models
Downloads a zip with `results/` (commit these to GitHub) and `artifacts/` (trained models).

In [28]:
!zip -qr results.zip results data/processed/dataset_stats.json data/processed/intent_stats.json
!zip -qr artifacts.zip artifacts data/processed/corpus.jsonl -x 'artifacts/*/checkpoints/*'
from google.colab import files
files.download("results.zip")
files.download("artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. (Optional) Deploy to Hugging Face Spaces
Create a token with *write* access at https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
SPACE = "Emmanuel-NS/farumasi-pharmacist-qa"
!python scripts/deploy_space.py --space $SPACE

/bin/bash: line 1: your-hf-username: No such file or directory
